# Python Spark SQL Exercises

For this set of exercises, you should use SQL statements, as
much as possible!

Check this online resource for some help with [SQL queries](https://www.codecademy.com/learn/learn-sql/modules/learn-sql-queries/cheatsheet)

In [ ]:
#@title Install Pyspark
!pip install --quiet pyspark unidecode
!apt-get install -y openjdk-17-jre 2>/dev/null > /dev/null

In [ ]:
#@title Download "Os Maias"
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0
!wc os_maias.txt

In [ ]:
#@title First Example
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]').appName('words').getOrCreate()

try :

  spark.read.text('os_maias.txt') \
      .withColumnRenamed('value', 'lines') \
      .createOrReplaceTempView("OSMAIAS")

  x = spark.sql("SELECT count(*) AS lines FROM OSMAIAS")
  x.show(5)
except Exception as err:
  print(err)

##1. Sorted Word Frequency

1.1) Create a [Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) program that counts the number of occurrences of each word in "Os Maias" novel;

a) No sorting required, words and frequency can appear in any order;

b) Sorted by word, in reverse alphabetical order;

c) Sorted by frequency (the words with higher occurrence first).

In [ ]:
#@title 1.1a)
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
import string
from unidecode import unidecode

spark = SparkSession.builder.master('local[*]').appName('words').getOrCreate()

def cleanup_lines( line ):
  return unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower().strip()

try :
  spark.udf.register("cleanupLines", cleanup_lines, StringType())

  spark.read.text('os_maias.txt') \
        .withColumnRenamed('value', 'lines') \
        .createOrReplaceTempView("OSMAIAS")

  x = spark.sql("SELECT word, count(*) AS frequency FROM \
                   (SELECT explode(split(cleanupLines(lines), ' ')) AS word FROM OSMAIAS) \
                 GROUP BY word")

  x.show(truncate=False)
except Exception as err:
  print(err)

In [ ]:
#@title 1.1b)
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
import string
from unidecode import unidecode

spark = SparkSession.builder.master('local[*]').appName('words').getOrCreate()

def cleanup_lines( line ):
  return unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower().strip()

try :
  spark.udf.register("cleanupLines", cleanup_lines, StringType())

  spark.read.text('os_maias.txt') \
        .withColumnRenamed('value', 'lines') \
        .createOrReplaceTempView("OSMAIAS")


  x = spark.sql("SELECT word, count(*) AS frequency FROM \
                   (SELECT explode(split(cleanupLines(lines),' ')) AS word FROM OSMAIAS) \
                 GROUP BY word ORDER BY word DESC")

  x.show()
except Exception as err:
  print(err)

In [ ]:
#@title 1.1c)
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
import string
from unidecode import unidecode

spark = SparkSession.builder.master('local[*]').appName('words').getOrCreate()

def cleanup_lines( line ):
  return unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower().strip()

try :
  spark.udf.register("cleanupLines", cleanup_lines, StringType())

  spark.read.text('os_maias.txt') \
        .withColumnRenamed('value', 'lines') \
        .createOrReplaceTempView("OSMAIAS")


  x = spark.sql("SELECT word, count(*) AS frequency FROM \
                   (SELECT explode(split(cleanupLines(lines),' ')) AS word FROM OSMAIAS) \
                 GROUP BY word ORDER BY frequency DESC")

  x.show()
except Exception as err:
  print(err)

1.2) Create a Spark Dataframes program that computes the top 10 most used words in "Os Maias" novel.

In [ ]:
#@title 1.2)
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
import string
from unidecode import unidecode

spark = SparkSession.builder.master('local[*]').appName('words').getOrCreate()

def cleanup_lines( line ):
  return unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower().strip()

try :
  spark.udf.register("cleanupLines", cleanup_lines, StringType())

  spark.read.text('os_maias.txt') \
        .withColumnRenamed('value', 'lines') \
        .createOrReplaceTempView("OSMAIAS")


  x = spark.sql("SELECT word, count(*) AS frequency FROM \
                   (SELECT explode(split(cleanupLines(lines),' ')) AS word FROM OSMAIAS) \
                 GROUP BY word ORDER BY frequency DESC\
                 LIMIT 10")

  x.show()
except Exception as err:
  print(err)

##2. Weblog Analysis

Consider a set of log files captured during a DDOS (*Distributed Denial of Service*) attack, containing information for the web accesses performed during the attack to the server.

The log files contain text lines as shown below, with TAB as the separator:

date |IP_source | status_code | operation | URL | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [ ]:
#@title Download the dataset
!wget -q -O web.log https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -1 web.log

!echo "date ipSource retValue op url time" > weblog_with_header.log
!cat web.log >> weblog_with_header.log
!head -2 weblog_with_header.log

2.1. Count the number of unique IP addresses involved in the DDOS attack.


In [ ]:
#@title 2.1
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]') \
						.appName('weblog').getOrCreate()

try :
    spark.read.csv('weblog_with_header.log', sep =' ', header=True, inferSchema=True) \
        .createOrReplaceTempView("WebLog")

    x = spark.sql("SELECT COUNT(DISTINCT ipSource) AS total_ips FROM WebLog")

    x.show()
except Exception as err:
    print(err)

2.2. For each interval of 10 seconds, provide the following information: [number of requests, average execution time, maximum time, minimum time]

In [ ]:
#@title 2.2
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]') \
						.appName('weblog').getOrCreate()

try :
    spark.read.csv('weblog_with_header.log', sep =' ', header=True, inferSchema=True) \
        .createOrReplaceTempView("WebLog")

    spark.sql('SELECT * FROM WebLog').printSchema()

    x = spark.sql("SELECT WINDOW(date, '10 seconds').start AS interval, \
                          COUNT(*) AS requests, \
                          AVG(time) AS avg_time, \
                          MAX(time) AS max_time, \
                          MIN(time) AS min_time  \
                          FROM WebLog \
                          GROUP BY interval ORDER BY interval")

    x.show()
except Exception as err:
    print(err)

2.3. Create an inverted index that, for each interval of 10 seconds, has a list of (unique) IPs executing accesses (to each URL).

In [ ]:
#@title 2.2
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]') \
						.appName('weblog').getOrCreate()

try :
    spark.read.csv('weblog_with_header.log', sep =' ', header=True, inferSchema=True) \
        .createOrReplaceTempView("WebLog")

    spark.sql('SELECT * FROM WebLog').printSchema()

    x = spark.sql("SELECT WINDOW(date, '10 seconds').start AS interval, url, \
                          COLLECT_SET( ipSource) AS ips\
                    FROM WebLog \
                    GROUP BY interval, url \
                    ORDER BY interval DESC, url ASC")

    x.show(truncate=False)
except Exception as err:
    print(err)